# Manual Verification of Ground Truth Matches

Review each match and decide if it's correct or needs to be marked as false positive.

In [ ]:
import json
from pathlib import Path
import pandas as pd

# Load ground truth, preferring the larger stratified sample
candidates = [
    Path('output/ground_truth_stratified_matches.json'),
    Path('output/ground_truth_matches.json'),
]
for candidate_path in candidates:
    if candidate_path.exists():
        ground_truth_path = candidate_path
        break
else:
    raise FileNotFoundError('No ground truth sample file found in output/')

with open(ground_truth_path, 'r', encoding='utf-8') as f:
    ground_truth = json.load(f)

print(f"📋 Loaded {len(ground_truth)} samples for verification from {ground_truth_path.name}")
print(f"\nReview goal: label each pair as CORRECT, FALSE POSITIVE, or UNCERTAIN")
print(f"\nLabels:")
print(f"  0 = CORRECT")
print(f"  1 = FALSE POSITIVE")
print(f"  ? = UNCERTAIN")
print(f"\nWhy this matters:")
print(f"  - correct pairs become positive training/evaluation data")
print(f"  - false positives become negative evidence")
print(f"  - uncertain pairs stay out until reviewed")

In [ ]:
# Initialize verification tracking
verification = {}

LABELS = {
    0: 'correct',
    1: 'false_positive',
    '?': 'uncertain',
}

def display_sample(sample_id):
    """Display a single sample for verification."""
    sample = ground_truth[sample_id - 1]

    print(f"\n{'='*80}")
    print(f"SAMPLE #{sample['sample_id']}")
    print(f"{'='*80}")

    # Metadata
    print(f"\n📅 Date Alignment:")
    print(f"   Enriched: {sample['enriched_date']}")
    print(f"   Flat:     {sample['flat_date']}")
    print(f"   Same day: {'✓ YES' if sample['is_same_day'] else '✗ NO'}")

    # Entity matches
    print(f"\n🏷️  Entity Matches:")
    print(f"   Places:         {sample['entities']['places']['matched_count']}/{sample['entities']['places']['total_in_enriched']}")
    print(f"   Persons:        {sample['entities']['persons']['matched_count']}/{sample['entities']['persons']['total_in_enriched']}")
    print(f"   Organizations:  {sample['entities']['organizations']['matched_count']}/{sample['entities']['organizations']['total_in_enriched']}")
    print(f"   Summary Anchor: {'🔗 YES' if sample['is_summary_anchor'] else '   NO'}")

    # Show actual found entity names when available
    for entity_type in ('places', 'persons', 'organizations'):
        enriched_items = sample['entities'][entity_type].get('enriched', [])
        found_items = sample['entities'][entity_type].get('found_in_flat', [])
        if enriched_items or found_items:
            print(f"   {entity_type.title()} found in flat: {', '.join(found_items) if found_items else 'none'}")

    # Confidence
    print(f"\n💯 Confidence Score: {sample['confidence_score']} (higher = more certain)")

    # Texts
    print(f"\n📄 ENRICHED TEXT (preview):")
    print(f"   {sample['enriched_preview'][:150]}...")
    print(f"\n📃 FLAT TEXT (preview):")
    print(f"   {sample['flat_preview'][:150]}...")

    current_label = verification.get(sample_id, 'not labeled')
    print(f"\n🏁 Current label: {current_label}")
    print(f"   Use verification[{sample_id}] = 0, 1, or '?'")

    return sample

# Display first sample
current_sample = display_sample(1)

In [ ]:
# Verification workflow
verification = {}

# Optional helper to label a sample

def label_sample(sample_id, label):
    if label not in (0, 1, '?'):
        raise ValueError("label must be 0, 1, or '?'")
    verification[sample_id] = label
    print(f"Set sample {sample_id} -> {LABELS[label]}")

print("👇 VERIFY SAMPLES BELOW")
print("\nInstructions:")
print("1. Run display_sample(N) to inspect sample N")
print("2. Compare enriched_preview with flat_preview")
print("3. Label with:")
print("   label_sample(N, 0)  -> CORRECT")
print("   label_sample(N, 1)  -> FALSE POSITIVE")
print("   label_sample(N, '?') -> UNCERTAIN")
print("\nExample:")
print("   label_sample(1, 0)")

In [ ]:
# View verification progress
print(f"\n📊 Verification Progress:")
print(f"Total samples:   {len(ground_truth)}")
print(f"Labeled:         {len(verification)}/{len(ground_truth)}")
print(f"\nResults so far:")

correct = sum(1 for v in verification.values() if v == 0)
false_pos = sum(1 for v in verification.values() if v == 1)
uncertain = sum(1 for v in verification.values() if v == '?')

print(f"  ✓ Correct:        {correct}")
print(f"  ✗ False positive: {false_pos}")
print(f"  ? Uncertain:      {uncertain}")
print(f"  ⏳ Not labeled:    {len(ground_truth) - len(verification)}")

In [ ]:
# SAVE VERIFICATION RESULTS
# Run this cell after completing your review

from pathlib import Path
import csv

# Split labels into separate lists
verified_correct = [sample for sample in ground_truth if verification.get(sample['sample_id']) == 0]
verified_wrong = [sample for sample in ground_truth if verification.get(sample['sample_id']) == 1]
verified_uncertain = [sample for sample in ground_truth if verification.get(sample['sample_id']) == '?']
unlabeled = [sample for sample in ground_truth if sample['sample_id'] not in verification]

# Save verified data
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

# JSON exports
correct_file = output_dir / 'ground_truth_correct.json'
wrong_file = output_dir / 'ground_truth_false_positives.json'
uncertain_file = output_dir / 'ground_truth_uncertain.json'
all_labels_file = output_dir / 'ground_truth_labels.json'

for file_path, data in [
    (correct_file, verified_correct),
    (wrong_file, verified_wrong),
    (uncertain_file, verified_uncertain),
]:
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

with open(all_labels_file, 'w', encoding='utf-8') as f:
    json.dump({str(k): v for k, v in verification.items()}, f, indent=2, ensure_ascii=False)

# Readable CSV summary
csv_file = output_dir / 'ground_truth_labels.csv'
with open(csv_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['sample_id', 'label', 'enriched_date', 'flat_date', 'same_day', 'summary_anchor', 'confidence_score'])
    for sample in ground_truth:
        label = verification.get(sample['sample_id'], '')
        writer.writerow([
            sample['sample_id'],
            LABELS.get(label, 'unlabeled'),
            sample['enriched_date'],
            sample['flat_date'],
            sample['is_same_day'],
            sample['is_summary_anchor'],
            sample['confidence_score'],
        ])

# Human-readable text report
report_file = output_dir / 'ground_truth_labels_report.txt'
with open(report_file, 'w', encoding='utf-8') as f:
    f.write('GROUND TRUTH LABEL REPORT\n')
    f.write('=' * 60 + '\n\n')
    f.write(f'Total samples: {len(ground_truth)}\n')
    f.write(f'Correct: {len(verified_correct)}\n')
    f.write(f'False positives: {len(verified_wrong)}\n')
    f.write(f'Uncertain: {len(verified_uncertain)}\n')
    f.write(f'Unlabeled: {len(unlabeled)}\n\n')
    f.write('Sample labels:\n')
    for sample in ground_truth:
        label = LABELS.get(verification.get(sample['sample_id']), 'unlabeled')
        f.write(f"- Sample {sample['sample_id']}: {label} | {sample['enriched_date']} -> {sample['flat_date']} | same_day={sample['is_same_day']} | summary_anchor={sample['is_summary_anchor']}\n")

print(f"✅ Verification Complete")
print(f"\n📊 Summary:")
print(f"  Total:            {len(ground_truth)}")
print(f"  ✓ Correct:        {len(verified_correct)}")
print(f"  ✗ False pos:      {len(verified_wrong)}")
print(f"  ? Uncertain:      {len(verified_uncertain)}")
print(f"  ⏳ Unlabeled:     {len(unlabeled)}")
print(f"\n💾 Saved:")
print(f"  - {correct_file}")
print(f"  - {wrong_file}")
print(f"  - {uncertain_file}")
print(f"  - {all_labels_file}")
print(f"  - {csv_file}")
print(f"  - {report_file}")